# 2D Gaussian

Run the six sampling methods using the local `PCA_Theta.py` module. The model and all experiment settings are defined below. `reference` mode loads existing results; `smoke` runs a short validation; `full` runs the paper experiment.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import os, json
import numpy as np
HERE = Path.cwd()
from PCA_Theta import METHODS, run_2d, save_result, load_result

## 1. Model

In [2]:
import numpy as np

EXAMPLE_NAME = "gaussian"
BOUNDS = [(0.0, 1.0), (0.0, 1.0)]


def gaussian_2d(x1, x2, c1, c2, s1, s2):
    return np.exp(-((x1 - c1) / s1) ** 2 - ((x2 - c2) / s2) ** 2)


def evaluate(X):
    X = np.asarray(X, dtype=float)
    x1, x2 = X[:, 0], X[:, 1]
    g0 = gaussian_2d(x1, x2, 0.25, 0.25, 0.16, 0.16)
    return np.column_stack([
        g0,
        1.03 * g0 + 0.02 * np.sin(2.0 * np.pi * x1),
        0.97 * g0 + 0.03 * np.cos(2.0 * np.pi * x2),
        g0 + 0.03 * np.sin(2.0 * np.pi * (x1 + x2)),
        0.85 * gaussian_2d(x1, x2, 0.75, 0.70, 0.13, 0.13),
        0.70 * gaussian_2d(x1, x2, 0.80, 0.25, 0.14, 0.14),
    ])


def build_model(args):
    return evaluate, {"n_outputs": 6}

## 2. Settings

In [3]:
# reference: inspect saved paper results; smoke: N=30 to 40, one repeat;
# full: use the paper settings below. Long runs begin only in full mode.
RUN_MODE = 'reference'
REPEATS = 50
VARIANTS = [('max', 'common'), ('loo', 'common')]
METHODS_TO_RUN = list(METHODS)
SETTINGS = dict(n_init=30, n_final=1000, batch_size=10,
                p_min=5, p_max=25, n_grid=100, n_test=1000,
                test_seed=2026, pca_threshold=0.01)
OUTPUT = Path(os.environ.get('PCA_THETA_OUTPUT', HERE/'Results'))

## 3. Run and save

In [4]:
if RUN_MODE not in ('reference', 'smoke', 'full'):
    raise ValueError('Choose reference, smoke, or full.')
if RUN_MODE == 'reference':
    print('Reference results are in Data/. Open Plotting.ipynb to reproduce the paper figures.')
else:
    settings = dict(SETTINGS)
    repetitions = REPEATS if RUN_MODE == 'full' else 1
    if RUN_MODE == 'smoke': settings['n_final'] = 40
    total = len(VARIANTS)*len(METHODS_TO_RUN)*repetitions
    done = 0
    for rule, mode in VARIANTS:
        for repeat in range(repetitions):
            model_args = SimpleNamespace(field_seed=123+1000*repeat,
                                         n_outputs=20, modes_per_dim=5, corr_decay=0.1)
            model, metadata = build_model(model_args)
            for method in METHODS_TO_RUN:
                path = OUTPUT/RUN_MODE/f'{rule}_{mode}_{method}_r{repeat:02d}.npz'
                if path.exists():
                    result = load_result(path)
                    if result.get('configuration') != settings:
                        raise ValueError('Saved settings differ; choose a new OUTPUT folder.')
                else:
                    result = run_2d(model, BOUNDS, method=method, degree_mode=mode,
                                    degree_rule=rule, repeat=repeat, **settings)
                    result['model_metadata'] = metadata
                    result['configuration'] = settings
                    save_result(path, result)
                done += 1
                print(f"[{done}/{total}] {rule}/{mode}, repeat={repeat+1}, {method}: "
                      f"N={result['hist']['n'][-1]}, NRMSE={result['hist']['nrmse'][-1]:.4g}", flush=True)

Reference results are in Data/. Open Plotting.ipynb to reproduce the paper figures.


## 4. Saved paper results

LHS uses a complete independent design at every budget after the shared initial ED. The plotted N is each design size, not the cumulative cost of all LHS designs. New runs are saved separately from Data/.

In [5]:
with np.load(HERE/'Data/results.npz', allow_pickle=False) as reference:
    print('Saved paper budgets:', reference['n'][0], 'to', reference['n'][-1])
    print('Statistics: 50 repetitions; plot means and standard deviations of log10 errors.')

Saved paper budgets: 30 to 1000
Statistics: 50 repetitions; plot means and standard deviations of log10 errors.
